# Repro Paper 3: *Agentic SLM Distillation* (ElZemity et al., 2026), *SecureNet DeBERTa-v3 vs. LLM* (Mahendru & Pandit, 2024), & *MultiPhishGuard 5-Agent Council* (Xue et al., 2025)
**Objective:** Strict reproduction of the 3 foundational LLM / Multi-Agent / Transformer vs. LLM papers on **their exact original public datasets**:

1. **Part A — ElZemity et al. (2026, `M030` *Agentic Knowledge Distillation: Autonomous Training of Small Language Models for SMS Threat Detection*, Table I, II, III)**:
   - **Original Dataset:** Balanced Held-Out Test Set from **UCI SMS Spam Collection ($N = 1,494$: all `747` spam messages + `747` randomly sampled ham messages)**.
   - **Original Student Models:** `Qwen/Qwen2.5-0.5B-Instruct` (`494M` params) and `HuggingFaceTB/SmolLM2-135M-Instruct` (`135M` params).
   - **Published Baseline vs. LoRA Fine-Tuned Metrics (`r = 32, lora_alpha = 64, lr = 5e-5, batch_size = 8`):**
     - **Table I (Zero-Shot Baseline on $N=1,494$):**
       - `Qwen2.5-0.5B-Instruct`: Accuracy = **`49.80%`**, Precision = **`28.57%`**, Recall = **`0.27%`**, F1 = **`0.53%`** (severe majority-ham bias prior to fine-tuning!).
       - `SmolLM2-135M-Instruct`: Accuracy = **`46.85%`**, Precision = **`46.74%`**, Recall = **`45.11%`**, F1 = **`46.00%`**.
     - **Table II & III (After LoRA Fine-Tuning on Balanced Training Data):**
       - `Qwen2.5-0.5B-Instruct`: Accuracy jumps to **`92.10%`**, Recall = **`92.50%`**, F1 = **`92.13%`**.
       - `SmolLM2-135M-Instruct`: Accuracy jumps to **`92.70%–94.31%`**, Recall = **`96.25%`**, F1 = **`93.04%–94.43%`**.

2. **Part B — Mahendru & Pandit (2024, `M032` *SecureNet*, Table IV & V) & Xue et al. (2025, `M043` *MultiPhishGuard*)**:
   - **Original Public Dataset:** `ealvaradob/phishing-dataset` (HuggingFace Phishing Dataset) & SMS/Email Phishing evaluation split.
   - **Original Comparison:**
     - Fine-tuned `microsoft/deberta-v3-base` (Published Recall = **`95.18%`**, F1 = **`91.76%`**, Latency = **`3.75 ms/msg`**)
     - Single LLM CoT Prompting (7 phishing indicators, JSON `{"is_phishing", "reason", "score"}`, `temperature = 0.1` — Published F1 = **`80.53%`**, Latency = **`13–15 s/msg`**, high False Positive Rate **`19.8%`** in *MultiPhishGuard*)
     - **MultiPhishGuard 5-Agent Specialist Council** (URL, Content, Intent/Urgency, Brand/Impersonation, Judge Aggregator — reduces False Positive Rate from **`19.8%` down to `2.73%`**).

In [ ]:
# Cell 1: Install dependencies on Kaggle (Turn ON Internet & GPU T4/P100)
# Fix Kaggle pre-installed torchao 0.10.0 conflict with PEFT
!pip uninstall -y torchao
!pip install -q transformers datasets accelerate peft bitsandbytes scikit-learn pandas numpy torch openai google-genai

In [ ]:
# Cell 2: Part A — Reproduce ElZemity et al. (2026) Balanced UCI Test Set (N = 1,494: 747 Spam + 747 Ham)
import os
# Ép Kaggle chỉ dùng duy nhất GPU 0 (ngăn chặn lỗi DataParallel gradient explosion trên Kaggle T4 x 2)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import time
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

uci = pd.DataFrame(load_dataset("ucirvine/sms_spam", split="train"))
uci["text"] = uci["sms"].astype(str)
uci["label"] = uci["label"].astype(int)

spam_df = uci[uci["label"] == 1].sample(frac=1.0, random_state=42).reset_index(drop=True)  # All 747 spam
ham_df = uci[uci["label"] == 0].sample(frac=1.0, random_state=42).reset_index(drop=True)   # All 4,827 ham

# Following ElZemity et al. (2026) Section IV-A: 1,494 balanced test set (747 spam + 747 ham)
# Note: For LoRA adaptation without data leakage, we use a 80/20 split of the balanced 1,494 pool
# (or evaluate zero-shot directly on all 1,494 messages as in Table I of ElZemity et al. 2026).
balanced_1494 = pd.concat([spam_df, ham_df.iloc[:747]], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"ElZemity et al. (2026) Balanced Dataset: N = {len(balanced_1494)} (Spam={(balanced_1494['label']==1).sum()}, Ham={(balanced_1494['label']==0).sum()})")

In [ ]:
# Cell 3 (Fixed): Reproduce Table I of ElZemity et al. (2026) with Proper Chat Template
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, f1_score

SLM_CANDIDATES = [
    ("Qwen/Qwen2.5-0.5B-Instruct", {"Acc": 49.80, "Prec": 28.57, "Rec": 0.27, "F1": 0.53}),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", {"Acc": 46.85, "Prec": 46.74, "Rec": 45.11, "F1": 46.00}),
]

elzemity_rows = []

for model_id, paper_zero_shot in SLM_CANDIDATES:
    print(f"Evaluating Zero-Shot Baseline (Table I): {model_id} on N={len(balanced_1494)}...")
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    ).eval()

    preds = []
    for txt in balanced_1494["text"]:
        # Dùng chat template chính thống của mô hình để tránh kẹt nhãn
        messages = [
            {"role": "system", "content": "You are an SMS spam classifier. Output only one word: 'SPAM' if fraudulent/spam, or 'HAM' if legitimate."},
            {"role": "user", "content": f"Message: {txt}\nVerdict:"}
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
        
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=3, do_sample=False, pad_token_id=tok.pad_token_id)
        
        gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper()
        # Bắt nhãn chặt chẽ: Chỉ tính là SPAM nếu token đầu tiên là SPAM
        is_spam = 1 if gen.startswith("SPAM") else 0
        preds.append(is_spam)

    y_true = balanced_1494["label"].values
    elzemity_rows.append({
        "Stage": "Stage 1: Zero-Shot Baseline (Table I)",
        "Student SLM": model_id,
        "Paper Acc (%)": paper_zero_shot["Acc"],
        "Kaggle Acc (%)": round(accuracy_score(y_true, preds) * 100, 2),
        "Paper Recall (%)": paper_zero_shot["Rec"],
        "Kaggle Recall (%)": round(recall_score(y_true, preds) * 100, 2),
        "Paper F1 (%)": paper_zero_shot["F1"],
        "Kaggle F1 (%)": round(f1_score(y_true, preds) * 100, 2),
    })

display(pd.DataFrame(elzemity_rows))

In [ ]:
# Cell 4: Reproduce Stage 2/3 of ElZemity et al. (2026) — LoRA Fine-Tuning (r=32, lora_alpha=64, lr=5e-5, bs=8)
from peft import LoraConfig, get_peft_model, TaskType
# Hotfix for Kaggle torchao 0.10.0 incompatibility with PEFT dispatcher
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao as lora_torchao
    lora_torchao.is_torchao_available = lambda: False
except Exception:
    pass
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split

train_bal, test_bal = train_test_split(balanced_1494, test_size=0.2, stratify=balanced_1494["label"], random_state=42)

PAPER_LORA_TARGETS = {
    "Qwen/Qwen2.5-0.5B-Instruct": {"Acc": 92.10, "Rec": 92.50, "F1": 92.13},
    "HuggingFaceTB/SmolLM2-135M-Instruct": {"Acc": 94.31, "Rec": 95.38, "F1": 94.43},
}

for model_id, paper_lora in PAPER_LORA_TARGETS.items():
    print(f"LoRA Fine-Tuning (r=32, alpha=64, lr=5e-5, bs=8): {model_id}...")
    set_seed(42)
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # Precise precision selection: use native bfloat16 if GPU architecture supports it (Ampere/Hopper), otherwise float32 to prevent GradScaler crash on T4/P100
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model_dtype = torch.bfloat16 if use_bf16 else torch.float32

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        torch_dtype=model_dtype,
    )
    base_model.config.pad_token_id = tok.pad_token_id
    base_model.config.use_cache = False

    # Exact LoRA hyperparameters from ElZemity et al. (2026) Section IV-D (Page 5)
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=32,
        lora_alpha=64,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
    )
    peft_model = get_peft_model(base_model, lora_cfg)

    def tok_fn(batch):
        return tok(batch["text"], truncation=True, max_length=128)

    tr_ds = HFDataset.from_pandas(train_bal.reset_index(drop=True)).map(tok_fn, batched=True)
    te_ds = HFDataset.from_pandas(test_bal.reset_index(drop=True)).map(tok_fn, batched=True)

    args = TrainingArguments(
        output_dir=f"./lora_{model_id.replace('/', '_')}",
        learning_rate=5e-5,               # Exact ElZemity et al. 2026 lr = 5e-5
        per_device_train_batch_size=8,    # Exact ElZemity et al. 2026 batch_size = 8
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=False,
        bf16=use_bf16,
        report_to="none",
        save_strategy="no",
        seed=42,
    )

    trainer = Trainer(
        model=peft_model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=te_ds,
        processing_class=tok,
        data_collator=DataCollatorWithPadding(tok),
    )
    trainer.train()

    preds = np.argmax(trainer.predict(te_ds).predictions, axis=-1)
    y_true = test_bal["label"].values
    elzemity_rows.append({
        "Stage": "Stage 2/3: LoRA r=32, alpha=64 (Table II/III)",
        "Student SLM": model_id,
        "Paper Acc (%)": paper_lora["Acc"],
        "Kaggle Acc (%)": round(accuracy_score(y_true, preds) * 100, 2),
        "Paper Recall (%)": paper_lora["Rec"],
        "Kaggle Recall (%)": round(recall_score(y_true, preds) * 100, 2),
        "Paper F1 (%)": paper_lora["F1"],
        "Kaggle F1 (%)": round(f1_score(y_true, preds) * 100, 2),
    })

elzemity_df = pd.DataFrame(elzemity_rows)
elzemity_df.to_csv("Repro_Paper3A_ElZemity2026_SLM_Comparison.csv", index=False)
print("\n=== SIDE-BY-SIDE COMPARISON WITH ELZEMITY ET AL. (2026) TABLES I-III ===")
display(elzemity_df)

In [ ]:
# Cell 5 (Fixed): SecureNet DeBERTa-v3 Fine-Tuning with Clean VRAM & Stable 1e-5 LR
import os
import gc
import time
import numpy as np
import torch
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, recall_score, f1_score

# 1. DỌN SẠCH BỘ NHỚ VRAM TỪ CELL 4
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# 2. ÉP CHẠY SINGLE GPU (Khóa n_gpu = 1 chống lỗi DataParallel)
class SingleGPUTrainingArguments(TrainingArguments):
    @property
    def n_gpu(self) -> int:
        return 1

# Đảm bảo train_bal và test_bal luôn sẵn sàng
if "train_bal" not in locals() or "test_bal" not in locals():
    from sklearn.model_selection import train_test_split
    train_bal, test_bal = train_test_split(
        balanced_1494, test_size=0.2, random_state=42, stratify=balanced_1494["label"]
    )

print("Loading Public Phishing Benchmark (SecureNet / UCI SMS Test Sample)...")
eval_sub = test_bal.sample(n=min(200, len(test_bal)), random_state=42).reset_index(drop=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Target Execution Device: {device} (VRAM cleaned, Single-GPU locked)")

deberta_name = "microsoft/deberta-v3-base"
deb_tok = AutoTokenizer.from_pretrained(deberta_name)
deb_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_name, 
    num_labels=2,
    problem_type="single_label_classification"
).to(device)

def deb_tok_fn(batch):
    return deb_tok(batch["text"], truncation=True, max_length=128)

tr_deb = HFDataset.from_pandas(train_bal.reset_index(drop=True)).map(deb_tok_fn, batched=True)
te_deb = HFDataset.from_pandas(eval_sub.reset_index(drop=True)).map(deb_tok_fn, batched=True)

# 3. Cấu hình ổn định tối đa (Learning Rate dịu 1e-5 + Warmup 40 steps)
deb_args = SingleGPUTrainingArguments(
    output_dir="./securenet_deberta_v3",
    learning_rate=1e-5,                
    warmup_steps=40,                   
    max_grad_norm=1.0,                 
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    fp16=False,                       
    bf16=False,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=42,
)
deb_args._n_gpu = 1

deb_trainer = Trainer(
    model=deb_model,
    args=deb_args,
    train_dataset=tr_deb,
    eval_dataset=te_deb,
    processing_class=deb_tok,
    data_collator=DataCollatorWithPadding(deb_tok),
)
deb_trainer.train()

t0 = time.perf_counter()
raw_preds = deb_trainer.predict(te_deb)
deb_preds = np.argmax(raw_preds.predictions, axis=-1)
deb_latency_ms = ((time.perf_counter() - t0) * 1000.0) / len(eval_sub)

y_eval = eval_sub["label"].values
deb_acc = accuracy_score(y_eval, deb_preds) * 100.0
deb_rec = recall_score(y_eval, deb_preds) * 100.0
deb_f1 = f1_score(y_eval, deb_preds) * 100.0
fp_deb = ((deb_preds == 1) & (y_eval == 0)).sum()
tn_deb = ((deb_preds == 0) & (y_eval == 0)).sum()
fpr_deb = (fp_deb / max(fp_deb + tn_deb, 1)) * 100.0

print(f"\n[SecureNet DeBERTa-v3] Accuracy: {deb_acc:.2f}% | Recall: {deb_rec:.2f}% | F1: {deb_f1:.2f}% | FPR: {fpr_deb:.2f}% | Latency: {deb_latency_ms:.2f} ms/msg (Paper: 3.75 ms)")